<a href="https://colab.research.google.com/github/Mjeed19/Athar/blob/main/Athar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Athar — 01. Data Loading

*Part of **Athar**, an AI-powered digital forensics system for fraud detection in the Enron email dataset.*

## Problem Definition

Enron collapsed in 2001 after one of the largest corporate fraud cases in history. During the investigation, around **517,000 internal emails** were made public.

Right now each email is stored as **one long block of raw text**. That is hard to search, filter, or analyse.

**Goal of this notebook:** load the raw emails and turn them into a structured table. This table is the foundation for the rest of the system.

### Contents
1. Import libraries
2. Load the raw data
3. Parse the emails into a table
4. Summary


## 1. Import Libraries

- **os** — builds file paths safely.
- **email** — Python's built-in library for reading raw email text.
- **pandas** — works with data as tables (DataFrames).
- **kagglehub** — downloads the dataset directly from Kaggle.

In [4]:
import os
import email

import pandas as pd
import kagglehub

## 2. Load the Raw Data

The dataset is downloaded from Kaggle. Anyone who runs this notebook gets the same data automatically.

We call the result `raw_df` to make it clear this is the data **before** any processing.

In [5]:
dataset_dir = kagglehub.dataset_download("wcukierski/enron-email-dataset")
raw_df = pd.read_csv(os.path.join(dataset_dir, "emails.csv"))

raw_df.shape

Using Colab cache for faster access to the 'enron-email-dataset' dataset.


(517401, 2)

In [6]:
raw_df.head()

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


**Observations**

- The data has **517,401 rows** and only **2 columns**:
  - `file`: where the email was stored (mailbox owner and folder).
  - `message`: the full raw email, with headers and body mixed together in one string.
- To analyse the emails, we first need to split `message` into separate fields.

## 3. Parse the Emails into a Table

Every raw email follows the same format: **headers** (`From:`, `To:`, `Date:`...) followed by the **body**.
The `email` library understands this format, so we use it to split each email into separate fields.

| Column | Why we need it |
|---|---|
| `message_id` | Unique ID for each email |
| `date` | When the email was sent |
| `sender`, `recipients`, `cc`, `bcc` | Who talked to whom |
| `subject`, `body` | The content |

In [7]:
def parse_email(raw_message):
    """Extract the main headers and body from one raw email."""
    msg = email.message_from_string(raw_message)
    return {
        "message_id": msg.get("Message-ID"),
        "date": msg.get("Date"),
        "sender": msg.get("From"),
        "recipients": msg.get("To"),
        "cc": msg.get("Cc"),
        "bcc": msg.get("Bcc"),
        "subject": msg.get("Subject"),
        "body": msg.get_payload(),
    }

In [8]:
emails_df = pd.DataFrame([parse_email(m) for m in raw_df["message"]])
emails_df["file"] = raw_df["file"]

emails_df.head()

,message_id,date,sender,recipients,cc,bcc,subject,body,file
0,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,None,None,,Here is our forecast\n\n,allen-p/_sent_mail/1.
1,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,None,None,Re:,Traveling to have a business meeting takes the...,allen-p/_sent_mail/10.
2,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,None,None,Re: test,test successful. way to go!!!,allen-p/_sent_mail/100.
3,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,None,None,,"Randy,\n\n Can you send me a schedule of the s...",allen-p/_sent_mail/1000.
4,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,None,None,Re: Hello,Let's shoot for Tuesday at 11:45.,allen-p/_sent_mail/1001.


In [9]:
emails_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   message_id  517401 non-null  object
 1   date        517401 non-null  object
 2   sender      517401 non-null  object
 3   recipients  495554 non-null  object
 4   cc          127881 non-null  object
 5   bcc         127881 non-null  object
 6   subject     517401 non-null  object
 7   body        517401 non-null  object
 8   file        517401 non-null  object
dtypes: object(9)
memory usage: 35.5+ MB
